# Google Colab Example for GPU-based MP2 correlation energy density generator

This notebook runs an example MP2 correlation energy density evaluation on Google Colab. It showcases the GPU-based implementation that uses the auxiliary basis expansion. 

## Table of contents 
- [Installation](#1-installation)
- [Molecule Setup](#2-molecule-setup)
- [Run `ec_mp2_gen`](#3-run-ec_mp2_gen)

## 1) Installation
For the Google Colab environment setup, select first a suitable runtime:

**Runtime → Change runtime type → Runtime Version → 2026.07**, with
any **NVIDIA GPU**. 

This runtime allows **Python 3.12** (see
[Colab's runtime list](https://research.google.com/colaboratory/runtime-version-faq.html)).

Installation uses the public GitHub repository and does not require a token. Set `REVISION` to a release tag to install that version. The repository must be public before this installation cell can run successfully.

Normal pip installs **CuPy, GPU4PySCF, cuTENSOR** and the pinned
numerical libraries. After installation, **restart the session once** before continuing.

In [ ]:
import os
import subprocess
import sys

REPOSITORY = "https://github.com/vuckovic-lab/MP2_edens_GPU.git"
REVISION = "1.0"  # Replace with release tag to install a fixed version.

if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Select Colab runtime 2026.07 (Python 3.12) with an NVIDIA GPU.")
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-input",
     f"git+{REPOSITORY}@{REVISION}"],
    check=True,
    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
)
print("Installation finished. Restart the session, then continue with section 2.")


## 2) Molecule setup

Choose your molecule system and its parameters for the PySCF evaluation. 
Note that PySCF uses `spin=2S`. The scf type can be `rhf` for a closed shell or `uhf` for an open shell system. 



In [ ]:
import time
import cupy as cp
from pyscf import gto
from gpu4pyscf import dft

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("Select an NVIDIA GPU runtime.")

struc = """
O   0.000000   0.000000   0.000000
H   0.000000  -0.757000   0.587000
H   0.000000   0.757000   0.587000
"""
charge = 0
spin = 0
basis = "def2qzvppd"
scf_type = "rhf"       # "rhf" or "uhf"
xc_fun = 'hf'

mol = gto.M(atom=struc, basis=basis, charge=charge, spin=spin,
            unit="Angstrom", cart=False, verbose=0)
if scf_type == "rhf":
    mf = dft.RKS(mol)
    mf.xc = xc_fun
elif scf_type == "uhf":
    mf = dft.UKS(mol)
    mf.xc = xc_fun
cp.cuda.runtime.deviceSynchronize()
scf_started = time.perf_counter()
mf.kernel()
cp.cuda.runtime.deviceSynchronize()
scf_seconds = time.perf_counter() - scf_started
print("HF energy / Hartree:", repr(float(mf.e_tot)))
print(f"SCF elapsed: {scf_seconds:.3f} s")


## 3) Run `ec_mp2_gen`
MP2 correlation energy density generation uses regularization `kappa=2.0` as the default. Choose the grid level and fitting basis independently of the orbital basis.

The returned `ec_out` object contains `tot` and `os` correlation energies in Hartree, `dens_tot` and `dens_os` energy densities per particle, and `rho`, `weights` and `coords` as float64 CuPy arrays.


In [ ]:
import time
import cupy as cp
from mp2_edens import ec_mp2_gen, make_grid

grid_level = 3
kappa = 2.0
auxbasis = basis + "-ri"
coords, weights = make_grid(mol, level=grid_level)

cp.cuda.runtime.deviceSynchronize()
mp2_started = time.perf_counter()
ec_out = ec_mp2_gen(mf, mol, coords, weights,
                     kappa=kappa, auxbasis=auxbasis, verbose=True)
cp.cuda.runtime.deviceSynchronize()
mp2_seconds = time.perf_counter() - mp2_started
print(f"κMP2 elapsed: {mp2_seconds:.3f} s")

print("Correlation energies / Hartree:")
print("Total:", repr(ec_out.tot))
print("OS:   ", repr(ec_out.os))

# All output arrays stay on the GPU. Reintegrate the total per-particle field:
energy_from_density = cp.sum(
    ec_out.rho * ec_out.weights * ec_out.dens_tot
).item()
print("Total from density integration:", repr(energy_from_density))
